# Week 4: Logistic Regression and Feature Scaling

This analysis builds upon previous regression models developed using the BRFSS diabetes dataset. While linear regression was used in earlier assignments, diabetes status is fundamentally a categorical outcome. Therefore, logistic regression may be a more appropriate modeling technique. This notebook evaluates logistic regression performance before and after feature scaling and examines the impact of scaling on model accuracy.

In [2]:
import pandas as pd
import numpy as np

diabetes = pd.read_csv(
    "diabetes_012_health_indicators_BRFSS2015.csv"
)

print("Dataset Shape:")
print(diabetes.shape)

diabetes.head()

Dataset Shape:
(253680, 22)


,Diabetes_012,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,5.0,18.0,15.0,1.0,0.0,9.0,4.0,3.0
1,0.0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,3.0,0.0,0.0,0.0,0.0,7.0,6.0,1.0
2,0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,5.0,30.0,30.0,1.0,0.0,9.0,4.0,8.0
3,0.0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,3.0,6.0
4,0.0,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,3.0,0.0,0.0,0.0,11.0,5.0,4.0


## Dataset Overview

The BRFSS diabetes dataset contains health, demographic, and lifestyle variables that may be associated with diabetes status. The target variable, Diabetes_012, indicates whether an individual has no diabetes, prediabetes, or diabetes.

In [3]:
X = diabetes.drop("Diabetes_012", axis=1)
y = diabetes["Diabetes_012"]

print("Feature Matrix Shape:", X.shape)
print("Target Vector Shape:", y.shape)

Feature Matrix Shape: (253680, 21)
Target Vector Shape: (253680,)


In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training Features:", X_train.shape)
print("Testing Features:", X_test.shape)
print("Training Labels:", y_train.shape)
print("Testing Labels:", y_test.shape)

Training Features: (202944, 21)
Testing Features: (50736, 21)
Training Labels: (202944,)
Testing Labels: (50736,)


## Train-Test Split

The dataset was divided into training and testing sets using an 80/20 split. The training data was used to fit the logistic regression model, while the testing data was reserved for evaluating model performance on unseen observations.

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

logreg = LogisticRegression(
    max_iter=1000
)

logreg.fit(X_train, y_train)

preds = logreg.predict(X_test)

print(
    "Accuracy:",
    accuracy_score(y_test, preds)
)

Accuracy: 0.8482734153263954


/usr/local/python/3.12.1/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## Logistic Regression Without Feature Scaling

A logistic regression model was first fit using the original feature values. The model achieved an accuracy of approximately 84.8% when predicting diabetes status.

However, the model generated a convergence warning, indicating that the optimization algorithm struggled to find the optimal solution within the maximum number of iterations. This issue is often caused by predictors existing on very different scales. For example, BMI, Age, and Income have different numeric ranges, which can make optimization more difficult for logistic regression.

These results motivate the use of feature scaling before fitting the model again.

## Feature Scaling

Feature scaling standardizes predictor variables so that each feature has a mean of 0 and a standard deviation of 1. This helps logistic regression converge more efficiently and prevents variables with larger numeric ranges from having an outsized influence during optimization.

In [7]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Feature scaling complete.")

Feature scaling complete.


In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

scaled_model = LogisticRegression(
    max_iter=1000
)

scaled_model.fit(
    X_train_scaled,
    y_train
)

scaled_preds = scaled_model.predict(
    X_test_scaled
)

scaled_accuracy = accuracy_score(
    y_test,
    scaled_preds
)

print("Scaled Accuracy:", scaled_accuracy)

Scaled Accuracy: 0.8482537054556922


## Logistic Regression With Feature Scaling

After standardizing the predictor variables, logistic regression was fit again using the scaled features. The scaled model achieved an accuracy of approximately 84.8%, which was nearly identical to the accuracy obtained using the unscaled features.

Although feature scaling did not significantly improve predictive performance, it improved the optimization process by placing all variables on a common scale. This is important because logistic regression relies on iterative optimization methods that can struggle when predictors have very different numeric ranges.

In [9]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    y_test,
    scaled_preds
)

print(cm)

[[41754     0  1041]
 [  871     0    73]
 [ 5714     0  1283]]


In [10]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test,
        scaled_preds
    )
)

              precision    recall  f1-score   support

         0.0       0.86      0.98      0.92     42795
         1.0       0.00      0.00      0.00       944
         2.0       0.54      0.18      0.27      6997

    accuracy                           0.85     50736
   macro avg       0.47      0.39      0.40     50736
weighted avg       0.80      0.85      0.81     50736



/usr/local/python/3.12.1/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/usr/local/python/3.12.1/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/usr/local/python/3.12.1/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape

## Confusion Matrix and Classification Report

Although the logistic regression model achieved an overall accuracy of approximately 84.8%, the confusion matrix revealed important limitations. The model performed very well when identifying individuals without diabetes, achieving a recall of 98% for class 0.

However, the model struggled to identify individuals with prediabetes and diabetes. Most notably, the model never predicted the prediabetes class, resulting in precision, recall, and F1-scores of zero for class 1. The model also identified only a small portion of individuals with diabetes, achieving a recall of approximately 18% for class 2.

These results suggest that accuracy alone is not sufficient for evaluating model performance on an imbalanced dataset. The model is heavily biased toward the majority class and has difficulty distinguishing between the less common diabetes categories.

## Conclusions

This analysis applied logistic regression to predict diabetes status using health, demographic, and lifestyle variables from the BRFSS diabetes dataset. Logistic regression was evaluated both before and after feature scaling.

The unscaled model achieved an accuracy of approximately 84.8% but generated convergence warnings during training. After applying feature scaling, the model achieved nearly identical accuracy while improving the optimization process and eliminating convergence concerns. This demonstrates that feature scaling can improve model training even when predictive performance remains unchanged.

Despite the relatively high overall accuracy, the confusion matrix and classification report revealed significant class imbalance issues. The model performed well when identifying individuals without diabetes but struggled to correctly classify prediabetes and diabetes cases. In particular, the model failed to predict any prediabetes observations, indicating that accuracy alone does not fully capture model effectiveness.

Overall, logistic regression provided a more appropriate framework for modeling diabetes status than the regression techniques explored in previous weeks. However, future analyses may benefit from techniques designed to better handle imbalanced classification problems, such as class weighting, resampling methods, support vector machines, or tree-based models.